<a href="https://colab.research.google.com/github/abdulhaseeb941/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulhaseeb941/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**My lane: `decline_recovery`**

I'm choosing the decline_recovery lane because it directly builds on what I already
found in Weeks 1-2. The starter pipeline defines a forward label — is_declining_label =
(trend_direction == "down") — and shows that a simple hand-written rule only reaches
about 24% precision at identifying which pages are declining, while a trained model
roughly triples that. That gap tells me this is a real, learnable problem: the signal
that predicts decline exists in the data, but it isn't obvious enough for a simple rule
to capture well. A content team could genuinely use a better early-warning system here,
rather than reacting to declines only after they're already severe.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess

REPO_URL = "https://github.com/abdulhaseeb941/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**The question:** Which pages currently ranking are likely to keep declining in organic
performance over the next period, early enough that someone can act?

**Unit of analysis:** a single content page (content_id) at a point in time — not a whole
client or domain.

**Output:** a ranked list of pages by decline risk, similar in shape to the baseline
refresh_queue the starter pipeline already produces.

**Decision it improves:** which pages a content/SEO team prioritizes for review or
refresh work this cycle, instead of guessing or reacting only after traffic has already
dropped sharply.

**Action someone takes:** an editor or strategist reviews the top-ranked pages first and
decides whether to update content, fix technical issues, or leave it alone.

**Cost of a wrong call:**
- False positive (flagged as declining, but it wasn't really): wastes editor time on a
page that didn't need attention.
- False negative (missed a real decliner): the page keeps losing traffic/visibility
unnoticed, which is more costly long-term since the team never even looks at it.

Because a missed decliner is harder to recover from than a wasted review, I'd rather the
model lean toward catching more true decliners even if it means a few extra false alarms.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. How big is the decline problem in this sample?
decline_rate = (df["trend_direction"] == "down").mean()
print(f"Share of pages currently trending down: {decline_rate:.1%}")

# 2. Is a simple hand-written rule already good enough, or is there room for a model?
print("Baseline hand-rule Precision@50 (from notebook 01): 0.240")
print("Trained model Precision@50 (from notebook 01): ~0.70-0.75")
print("-> the model finds roughly 3x more true decliners in the top 50 than the rule")

# 3. Does a simple, obvious signal (staleness) already separate decliners from the rest?
avg_days_decline = df.loc[df["trend_direction"] == "down", "days_since_last_update"].mean()
avg_days_other  = df.loc[df["trend_direction"] != "down", "days_since_last_update"].mean()
print(f"Avg days since last update — declining pages: {avg_days_decline:.0f}")
print(f"Avg days since last update — non-declining pages: {avg_days_other:.0f}")


Share of pages currently trending down: 54.2%
Baseline hand-rule Precision@50 (from notebook 01): 0.240
Trained model Precision@50 (from notebook 01): ~0.70-0.75
-> the model finds roughly 3x more true decliners in the top 50 than the rule
Avg days since last update — declining pages: 49
Avg days since last update — non-declining pages: 42


**What this tells me:** over half of the sample (54.2%) is currently declining, so this
isn't a niche problem — it's the default state for many pages. A single obvious signal
like content staleness barely separates decliners from the rest (49 vs. 42 days on
average), which rules out a trivial fix and is exactly why a model that combines several
features is worth building. The pipeline's own precision numbers back this up: a simple
hand-written rule only reaches 24% precision on its top-50 picks, while a trained model
reaches roughly 70-75% — a real, ~3x improvement. (Note: the 54.2% figure is the overall
decline rate across the full sample, while the precision numbers come from the pipeline's
own train/test split, so they aren't directly comparable base rates — just two separate
pieces of evidence pointing the same direction.)

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can say:** the numbers above are observed and directional, not causal. This work
can identify pages whose combination of signals (position, staleness, engagement, etc.)
looks similar to pages that have historically declined, and use that to rank pages by
risk — supporting a human decision about where to look first. It can say "this page
resembles others that declined" and "this feature combination improves precision over a
simple rule."

**What I can't say:** I can't claim this predicts what Google's algorithm will do, and I
can't claim a flagged page will definitely decline — only that it shares patterns with
pages that have. This model doesn't establish causation (e.g., staleness doesn't
necessarily cause decline, it's just correlated with it in this sample). It also can't
promise a fix works — recovery is a separate, harder question from prediction, and
"decline_recovery" as a lane name doesn't mean the model tells anyone how to recover, only
who to look at first. Finally, all numbers here come from a small anonymized sample
(~30k rows) and an in-sample evaluation in places, so they may shift once I validate
properly with a held-out split on the full warehouse data in later weeks.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.